# Análisis Estadístico — Dataset de Deserción Estudiantil IFE
## TC3002B · Desarrollo de Aplicaciones Avanzadas · Tecnológico de Monterrey
**Autor:** Víctor Misael Escalante Alvarado  
**Dataset:** Alvarado-Uribe et al. (2022) — IFE Student Dropout  
**Descripción:** Notebook reproducible que cubre las secciones II (Análisis Descriptivo),
III (Análisis Multivariado) y IV (Validación Estadística del Modelado) del trabajo final.

---
### Prerrequisitos
Ejecutar antes:
1. `data/preprocessing/02_imputation_pipeline.ipynb` → genera `df_preprocessed.csv`
2. `analysis/clustering/02_kmeans_independiente.ipynb` → genera `df_pre.csv`, `df_tec.csv`
3. `analysis/modelos/experimentos/random_forest.ipynb` → genera `rf_model.pkl`, `X_te_tec.npy`, `y_te_tec.npy`

O bien, usar directamente los archivos ya procesados en `data/processed/`.

## 0. Setup e Importaciones

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import pickle
import json
from pathlib import Path

from scipy import stats
from scipy.stats import (
    spearmanr, mannwhitneyu, kruskal,
    kstest, anderson, norm, beta, gamma, lognorm
)
from sklearn.metrics import (
    roc_auc_score, recall_score, f1_score, precision_score,
    confusion_matrix, ConfusionMatrixDisplay
)

# ── Rutas ─────────────────────────────────────────────────────────────────
# Si ejecutas desde analysis/modelos/, las rutas relativas funcionan.
# Para ejecutar desde la raíz del proyecto, usa ruta absoluta:
#   BASE_DIR = Path('data/processed')
BASE_DIR      = Path('../../data/processed')
FIGURES_DIR   = Path('../../results/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
np.random.seed(SEED)

# Estilo visual
plt.rcParams.update({
    'figure.dpi': 150,
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
})
sns.set_style('whitegrid')

print('✓ Setup completado')

## 1. Carga de Datos

In [ ]:
df = pd.read_csv(BASE_DIR / 'df_preprocessed.csv')
print(f'Dataset cargado: {df.shape[0]:,} estudiantes × {df.shape[1]} variables')
print(f'Régimen: {dict(df["regime"].value_counts())}')
print(f'Retención (1=retenido, 0=desertor): {dict(df["retention"].value_counts())}')

# Tasa de deserción global y por régimen
dropout_global = 1 - df['retention'].mean()
dropout_pre    = 1 - df[df['regime']=='PreTec21']['retention'].mean()
dropout_tec    = 1 - df[df['regime']=='Tec21']['retention'].mean()
print(f'\nTasa de deserción  — global: {dropout_global:.1%}  |  PreTec21: {dropout_pre:.1%}  |  Tec21: {dropout_tec:.1%}')

# Variables clave de análisis
VARS_CLAVE = ['PNA', 'general.math.eval', 'admission_test_norm',
              'FTE', 'apoyo_financiero']

df_pre = df[df['regime'] == 'PreTec21'].copy()
df_tec = df[df['regime'] == 'Tec21'].copy()
print(f'\ndf_pre: {df_pre.shape}  |  df_tec: {df_tec.shape}')

---
## Sección II — Análisis Descriptivo

### 2.1 Estadísticos Descriptivos

In [ ]:
desc = df[VARS_CLAVE].describe().T
desc['skewness'] = df[VARS_CLAVE].skew()
desc['kurtosis'] = df[VARS_CLAVE].kurt()
desc['missing_%'] = df[VARS_CLAVE].isna().mean() * 100
print(desc[['count','mean','std','min','25%','50%','75%','max','skewness','kurtosis','missing_%']].to_string())

### 2.2 Visualización: Histogramas con KDE y Curva Normal

In [ ]:
VARS_HIST = ['PNA', 'general.math.eval', 'admission_test_norm']
LABELS = {'PNA': 'PNA', 'general.math.eval': 'Eval. Mat. General',
          'admission_test_norm': 'Prueba de Admisión (norm.)'}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, var in zip(axes, VARS_HIST):
    data = df[var].dropna()
    ax.hist(data, bins=50, density=True, alpha=0.55, color='#2563eb', label='Datos')

    # KDE real
    kde = stats.gaussian_kde(data)
    xgrid = np.linspace(data.min(), data.max(), 300)
    ax.plot(xgrid, kde(xgrid), lw=2, color='#1e40af', label='KDE')

    # Curva normal teórica
    mu, sigma = data.mean(), data.std()
    ax.plot(xgrid, norm.pdf(xgrid, mu, sigma), lw=2, ls='--',
            color='#dc2626', label=f'Normal(μ={mu:.2f}, σ={sigma:.2f})')

    ax.set_title(LABELS[var])
    ax.set_xlabel(LABELS[var])
    ax.set_ylabel('Densidad')
    ax.legend(fontsize=7)

plt.suptitle('Distribuciones con KDE y Normal Teórica', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_hist_kde.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_hist_kde.png guardado')

### 2.3 Gráficas Q-Q (Normal)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, var in zip(axes, VARS_HIST):
    data = df[var].dropna()
    (osm, osr), (slope, intercept, r) = stats.probplot(data, dist='norm')
    ax.scatter(osm, osr, alpha=0.15, s=4, color='#2563eb', label='Observado')
    line_x = np.array([min(osm), max(osm)])
    ax.plot(line_x, slope * line_x + intercept, lw=2, color='#dc2626', label=f'r={r:.4f}')
    ax.set_title(f'Q-Q Normal: {LABELS[var]}')
    ax.set_xlabel('Cuantiles Teóricos')
    ax.set_ylabel('Cuantiles Muestrales')
    ax.legend(fontsize=8)

plt.suptitle('Gráficas Q-Q (Distribución Normal)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_qqplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_qqplots.png guardado')

### 2.4 Boxplots por Régimen

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 4))

for ax, var in zip(axes, VARS_HIST):
    data_plot = df[['regime', var]].dropna()
    pre_vals = data_plot[data_plot['regime']=='PreTec21'][var]
    tec_vals = data_plot[data_plot['regime']=='Tec21'][var]

    bp = ax.boxplot([pre_vals, tec_vals],
                    labels=['PreTec21', 'Tec21'],
                    patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    bp['boxes'][0].set_facecolor('#93c5fd')
    bp['boxes'][1].set_facecolor('#fca5a5')

    ax.set_title(LABELS[var])
    ax.set_ylabel(LABELS[var])

plt.suptitle('Distribución por Régimen (PreTec21 vs Tec21)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_boxplots.png guardado')

### 2.5 Pruebas de Normalidad
Para n=77,517, las pruebas estándar rechazan la normalidad trivialmente.
Reportamos tanto el estadístico como el p-valor; el foco está en la magnitud.

In [ ]:
print('=== Pruebas de Normalidad ===\n')
print(f'{"Variable":<28} {"KS stat":>8} {"KS p":>10} {"AD stat":>10} {"AD crit(5%)":>12} {"Rechaza?":>10}')
print('-'*80)

norm_results = {}
for var in VARS_CLAVE:
    data = df[var].dropna().values
    mu, sigma = data.mean(), data.std()

    # Kolmogorov-Smirnov vs Normal
    ks_stat, ks_p = kstest(data, 'norm', args=(mu, sigma))

    # Anderson-Darling
    ad_result = anderson(data, dist='norm')
    ad_stat   = ad_result.statistic
    ad_crit5  = ad_result.critical_values[2]   # 5% significance level
    reject    = 'Sí' if ad_stat > ad_crit5 else 'No'

    print(f'{var:<28} {ks_stat:>8.4f} {ks_p:>10.2e} {ad_stat:>10.2f} {ad_crit5:>12.3f} {reject:>10}')
    norm_results[var] = dict(ks_stat=ks_stat, ks_p=ks_p, ad_stat=ad_stat, ad_crit5=ad_crit5)

print('\nNota: con n=77,517 cualquier desviación leve produce p≈0. La magnitud del\n'
      'estadístico AD es más informativa: PNA (AD≈157) vs apoyo_financiero (AD≈6616).')

### 2.6 Ajuste de Distribuciones (MLE + AIC/BIC)

In [ ]:
from scipy.stats import norm as sp_norm, beta as sp_beta, gamma as sp_gamma, lognorm as sp_lognorm

DIST_CANDIDATES = {
    'Normal':    sp_norm,
    'Beta':      sp_beta,
    'Gamma':     sp_gamma,
    'Log-Normal': sp_lognorm,
}

def aic_bic(logL, k, n):
    aic = 2*k - 2*logL
    bic = k*np.log(n) - 2*logL
    return aic, bic

def fit_distributions(data, varname, clip01=False):
    """Ajusta múltiples distribuciones por MLE y compara AIC/BIC."""
    data = data.dropna().values
    if clip01:
        eps = 1e-6
        data = np.clip(data, eps, 1-eps)
    n = len(data)
    rows = []
    for name, dist in DIST_CANDIDATES.items():
        try:
            params = dist.fit(data)
            k = len(params)
            logL = dist.logpdf(data, *params).sum()
            a, b = aic_bic(logL, k, n)
            rows.append({'Distribución': name, 'Params': params, 'LogL': logL, 'AIC': a, 'BIC': b})
        except Exception as e:
            rows.append({'Distribución': name, 'Params': None, 'LogL': np.nan, 'AIC': np.nan, 'BIC': np.nan})
    res = pd.DataFrame(rows).sort_values('AIC')
    print(f'\n── {varname} (n={n:,}) ──')
    print(res[['Distribución','LogL','AIC','BIC']].to_string(index=False))
    return res

print('=== Ajuste de Distribuciones por MLE ===')
fit_results = {}
fit_results['PNA']                 = fit_distributions(df['PNA'],                'PNA')
fit_results['general.math.eval']   = fit_distributions(df['general.math.eval'],  'Eval. Mat. General')
fit_results['admission_test_norm'] = fit_distributions(df['admission_test_norm'], 'Prueba Admisión (norm.)', clip01=True)
fit_results['apoyo_financiero']    = fit_distributions(df['apoyo_financiero'],    'Apoyo Financiero', clip01=True)

### 2.7 Distribuciones Acotadas: Beta + FTE

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# ── Apoyo Financiero: Beta fit ──
ax = axes[0]
data_af = df['apoyo_financiero'].dropna().values
data_af_clip = np.clip(data_af, 1e-6, 1-1e-6)
ax.hist(data_af, bins=50, density=True, alpha=0.5, color='#2563eb', label='Datos')
try:
    a_b, b_b, loc_b, scale_b = sp_beta.fit(data_af_clip, floc=0, fscale=1)
    xg = np.linspace(0, 1, 300)
    ax.plot(xg, sp_beta.pdf(xg, a_b, b_b, loc=0, scale=1), lw=2,
            color='#dc2626', label=f'Beta(α={a_b:.2f}, β={b_b:.2f})')
except Exception:
    pass
ax.set_title('Apoyo Financiero (variable [0,1])')
ax.set_xlabel('Proporción'); ax.set_ylabel('Densidad'); ax.legend()

# ── FTE: distribución bimodal ──
ax = axes[1]
data_fte = df['FTE'].dropna().values
ax.hist(data_fte, bins=60, density=True, alpha=0.5, color='#16a34a', label='Datos')
kde_fte = stats.gaussian_kde(data_fte)
xg = np.linspace(data_fte.min(), data_fte.max(), 300)
ax.plot(xg, kde_fte(xg), lw=2, color='#15803d', label='KDE')
ax.set_title('FTE (Tiempo Completo Equivalente)')
ax.set_xlabel('FTE'); ax.set_ylabel('Densidad'); ax.legend()

plt.suptitle('Variables Acotadas y Distribuciones Ajustadas', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_apoyo_fte.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_apoyo_fte.png guardado')

### 2.8 Análisis de Valores Atípicos (IQR)

In [ ]:
print('=== Análisis de Outliers por IQR ===\n')
print(f'{"Variable":<28} {"Q1":>8} {"Q3":>8} {"IQR":>8} {"LB":>10} {"UB":>10} {"N outliers":>12} {"% total":>8}')
print('-'*90)

for var in VARS_CLAVE:
    data = df[var].dropna()
    q1, q3 = data.quantile(0.25), data.quantile(0.75)
    iqr = q3 - q1
    lb  = q1 - 1.5*iqr
    ub  = q3 + 1.5*iqr
    n_out = ((data < lb) | (data > ub)).sum()
    pct   = n_out / len(data) * 100
    print(f'{var:<28} {q1:>8.3f} {q3:>8.3f} {iqr:>8.3f} {lb:>10.3f} {ub:>10.3f} {n_out:>12,} {pct:>7.1f}%')

print('\nNota: FTE tiene IQR estrecho (~0.08) → el 16% marcado como outlier refleja la\n'
      'distribución discreta de cargas, no errores de medición.')

### 2.9 Desbalance de Clases y Tasa por Régimen

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart de retención global
ax = axes[0]
counts = df['retention'].value_counts().sort_index()
ax.bar(['Desertor (0)', 'Retenido (1)'], counts.values,
       color=['#dc2626','#16a34a'], alpha=0.8)
for i, v in enumerate(counts.values):
    ax.text(i, v + 300, f'{v:,}\n({v/len(df):.1%})', ha='center', fontsize=9)
ax.set_ylabel('Número de Estudiantes')
ax.set_title('Distribución Global de Retención')

# Tasa de deserción por régimen
ax = axes[1]
rates = df.groupby('regime')['retention'].apply(lambda x: 1-x.mean())
rates = rates.reindex(['PreTec21','Tec21'])
bars = ax.bar(rates.index, rates.values, color=['#93c5fd','#fca5a5'], alpha=0.9)
for bar, val in zip(bars, rates.values):
    ax.text(bar.get_x() + bar.get_width()/2, val + 0.002, f'{val:.1%}',
            ha='center', fontsize=10, fontweight='bold')
ax.set_ylabel('Tasa de Deserción')
ax.set_title('Tasa de Deserción por Régimen')
ax.set_ylim(0, rates.max() * 1.25)

plt.suptitle('Desbalance de Clases', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_retention_dist.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_retention_dist.png guardado')

---
## Sección III — Análisis Multivariado

### 3.1 Correlación de Spearman

In [ ]:
VARS_CORR = ['PNA', 'general.math.eval', 'admission_test_norm', 'FTE', 'apoyo_financiero']

df_corr = df[VARS_CORR].dropna()
n_corr  = len(df_corr)

# Matriz de correlaciones + p-valores
corr_matrix  = np.zeros((len(VARS_CORR), len(VARS_CORR)))
pval_matrix  = np.zeros_like(corr_matrix)
for i, v1 in enumerate(VARS_CORR):
    for j, v2 in enumerate(VARS_CORR):
        r, p = spearmanr(df_corr[v1], df_corr[v2])
        corr_matrix[i, j] = r
        pval_matrix[i, j] = p

labels = ['PNA','Mat. General','Adm. Test','FTE','Apoyo Fin.']

fig, ax = plt.subplots(figsize=(7, 5))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='RdBu_r',
            vmin=-1, vmax=1, mask=mask, linewidths=0.5,
            xticklabels=labels, yticklabels=labels, ax=ax,
            annot_kws={'size':9})
ax.set_title(f'Correlación de Spearman (n={n_corr:,})')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_spearman_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_spearman_heatmap.png guardado')

# Tabla de resultados
print('\n=== Correlaciones Spearman significativas (|r|>0.15, p<0.001) ===')
for i in range(len(VARS_CORR)):
    for j in range(i+1, len(VARS_CORR)):
        r = corr_matrix[i,j]; p = pval_matrix[i,j]
        if abs(r) > 0.15 and p < 0.001:
            print(f'  {VARS_CORR[i]:25s} ~ {VARS_CORR[j]:25s} : r={r:+.4f}, p={p:.2e}')

### 3.2 Comparación entre Regímenes: Mann-Whitney U

In [ ]:
def rank_biserial(u_stat, n1, n2):
    """Correlación biserial de rango (tamaño de efecto para Mann-Whitney)."""
    return 1 - (2 * u_stat) / (n1 * n2)

def cohens_d(a, b):
    na, nb = len(a), len(b)
    pooled_std = np.sqrt(((na-1)*a.std()**2 + (nb-1)*b.std()**2) / (na+nb-2))
    return (a.mean() - b.mean()) / pooled_std if pooled_std > 0 else 0

print('=== Mann-Whitney U: PreTec21 vs Tec21 ===\n')
print(f'{"Variable":<28} {"U stat":>12} {"p-valor":>10} {"r_rb":>8} {"Cohen d":>9} {"Sig?":>6}')
print('-'*80)

mw_results = {}
for var in VARS_CLAVE:
    pre_d = df_pre[var].dropna().values
    tec_d = df_tec[var].dropna().values
    u, p  = mannwhitneyu(pre_d, tec_d, alternative='two-sided')
    r_rb  = rank_biserial(u, len(pre_d), len(tec_d))
    d     = cohens_d(pre_d, tec_d)
    sig   = '***' if p < 0.001 else ('**' if p < 0.01 else ('*' if p < 0.05 else 'ns'))
    print(f'{var:<28} {u:>12.0f} {p:>10.2e} {r_rb:>8.4f} {d:>9.4f} {sig:>6}')
    mw_results[var] = dict(U=u, p=p, r_rb=r_rb, d=d)

print('\nConvención tamaño efecto |r_rb|: pequeño≥0.10, mediano≥0.30, grande≥0.50')

### 3.3 Prueba Kruskal-Wallis por Generación

In [ ]:
def eta_squared_kruskal(h_stat, n, k):
    """η² para Kruskal-Wallis (aproximación)."""
    return (h_stat - k + 1) / (n - k)

print('=== Kruskal-Wallis por Generación (AD14–AD20) ===\n')
print(f'{"Variable":<28} {"H stat":>8} {"p-valor":>10} {"η²":>8} {"df":>4}')
print('-'*65)

gens = df['generation'].dropna().unique()
for var in VARS_CLAVE:
    groups = [df[df['generation']==g][var].dropna().values for g in gens]
    groups = [g for g in groups if len(g) > 10]
    if len(groups) < 2:
        continue
    h, p = kruskal(*groups)
    n_tot = sum(len(g) for g in groups)
    k     = len(groups)
    eta2  = eta_squared_kruskal(h, n_tot, k)
    print(f'{var:<28} {h:>8.2f} {p:>10.2e} {eta2:>8.4f} {k-1:>4}')

### 3.4 Comparación Desertor vs Retenido (Boxplots y Pruebas)

In [ ]:
VARS_CLASS = ['PNA', 'general.math.eval', 'FTE']
LABELS_CLS = {'PNA':'PNA','general.math.eval':'Eval. Mat.','FTE':'FTE'}

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
print('=== Mann-Whitney U: Desertor vs Retenido ===\n')

for ax, var in zip(axes, VARS_CLASS):
    drop_d    = df[df['retention']==0][var].dropna().values
    retain_d  = df[df['retention']==1][var].dropna().values
    u, p      = mannwhitneyu(drop_d, retain_d, alternative='two-sided')
    r_rb      = rank_biserial(u, len(drop_d), len(retain_d))
    sig       = '***' if p < 0.001 else ('**' if p < 0.01 else '*')
    print(f'  {var:<25}  U={u:.0f}, p={p:.2e}, r_rb={r_rb:.4f} {sig}')

    bp = ax.boxplot([drop_d, retain_d], labels=['Desertor','Retenido'],
                    patch_artist=True,
                    medianprops=dict(color='black', lw=2))
    bp['boxes'][0].set_facecolor('#fca5a5')
    bp['boxes'][1].set_facecolor('#86efac')
    ax.set_title(f'{LABELS_CLS[var]}\n(p={p:.1e}, r_rb={r_rb:.3f})')

plt.suptitle('Variables Clave: Desertor vs Retenido', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_class_boxplots.png', dpi=150, bbox_inches='tight')
plt.show()
print('\n✓ fig_class_boxplots.png guardado')

### 3.5 Análisis Estructural del Régimen Tec21

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Eval. Matemáticas por régimen
ax = axes[0]
for regime, color, ls in [('PreTec21','#2563eb','-'),('Tec21','#dc2626','--')]:
    data = df[df['regime']==regime]['general.math.eval'].dropna()
    ax.hist(data, bins=40, density=True, alpha=0.35, color=color, label=regime)
    kde = stats.gaussian_kde(data)
    xg  = np.linspace(data.min(), data.max(), 300)
    ax.plot(xg, kde(xg), lw=2, ls=ls, color=color)
ax.set_title('Evaluación Matemática por Régimen')
ax.set_xlabel('Puntuación'); ax.set_ylabel('Densidad'); ax.legend()

# FTE por régimen
ax = axes[1]
for regime, color, ls in [('PreTec21','#2563eb','-'),('Tec21','#dc2626','--')]:
    data = df[df['regime']==regime]['FTE'].dropna()
    ax.hist(data, bins=40, density=True, alpha=0.35, color=color, label=regime)
    kde = stats.gaussian_kde(data)
    xg  = np.linspace(data.min(), data.max(), 300)
    ax.plot(xg, kde(xg), lw=2, ls=ls, color=color)
ax.set_title('FTE por Régimen')
ax.set_xlabel('FTE'); ax.set_ylabel('Densidad'); ax.legend()

plt.suptitle('Cambios Estructurales: PreTec21 → Tec21', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_regime_structural.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_regime_structural.png guardado')

---
## Sección IV — Validación Estadística del Modelado

### 4.1 Carga del Modelo y Datos de Prueba

In [ ]:
# Carga del modelo Random Forest entrenado en PreTec21
with open(BASE_DIR / 'rf_model.pkl', 'rb') as f:
    rf_model = pickle.load(f)

# Arrays de test (Tec21) — ya preprocesados (n=4,902, 26 features)
X_te = np.load(BASE_DIR / 'X_te_tec.npy')
y_te = np.load(BASE_DIR / 'y_te_tec.npy')

with open(BASE_DIR / 'feat_pre.json') as f:
    feat_names = json.load(f)

print(f'Modelo: {rf_model}')
print(f'X_te: {X_te.shape}  |  y_te: {y_te.shape}')
print(f'Features ({len(feat_names)}): {feat_names[:5]} ...')
print(f'Desertores en test: {(y_te==0).sum()} ({(y_te==0).mean():.1%})')

from sklearn.metrics import precision_recall_curve

# y_proba_dropout: probabilidad de deserción (clase 0)
y_proba_dropout = rf_model.predict_proba(X_te)[:, 0]
y_te_bin        = (y_te == 0).astype(int)    # 1=desertor, 0=retenido

# Umbral óptimo por índice de Youden (maximiza TPR - FPR)
from sklearn.metrics import roc_curve
fpr, tpr, thresholds_roc = roc_curve(y_te_bin, y_proba_dropout)
j_scores   = tpr - fpr
opt_idx    = np.argmax(j_scores)
opt_thresh = thresholds_roc[opt_idx]

y_pred_bin = (y_proba_dropout >= opt_thresh).astype(int)   # 1=predicho desertor

auc_val  = roc_auc_score(y_te_bin, y_proba_dropout)
recall_v = recall_score(y_te_bin, y_pred_bin, zero_division=0)
prec_v   = precision_score(y_te_bin, y_pred_bin, zero_division=0)
f1_v     = f1_score(y_te_bin, y_pred_bin, zero_division=0)

print(f'Umbral óptimo (Youden J): {opt_thresh:.4f}')
print(f'AUC-ROC  : {auc_val:.4f}')
print(f'Recall   : {recall_v:.4f}')
print(f'Precision: {prec_v:.4f}')
print(f'F1-Score : {f1_v:.4f}')

In [ ]:
from sklearn.metrics import precision_recall_curve

y_proba = rf_model.predict_proba(X_te)[:, 1]

# Buscar umbral que maximice F1 manteniendo Recall≥0.60
prec_arr, rec_arr, thresh_arr = precision_recall_curve(y_te, y_proba, pos_label=0)

# Alternativa: buscar umbral óptimo por Youden's J
from sklearn.metrics import roc_curve
fpr, tpr, thresholds_roc = roc_curve(y_te, y_proba, pos_label=0)
j_scores  = tpr - fpr
opt_idx   = np.argmax(j_scores)
opt_thresh = thresholds_roc[opt_idx]

y_pred = (y_proba <= opt_thresh).astype(int)   # clase 0 = desertor

auc_val  = roc_auc_score(y_te, y_proba, pos_label=0, multi_class='ovr')
recall_v = recall_score(y_te, y_pred, pos_label=0)
prec_v   = precision_score(y_te, y_pred, pos_label=0)
f1_v     = f1_score(y_te, y_pred, pos_label=0)

print(f'Umbral óptimo (Youden J): {opt_thresh:.4f}')
print(f'AUC-ROC  : {auc_val:.4f}')
print(f'Recall   : {recall_v:.4f}')
print(f'Precision: {prec_v:.4f}')
print(f'F1-Score : {f1_v:.4f}')

### 4.3 Bootstrap — Intervalos de Confianza (B=1,000)

In [ ]:
B = 1_000
rng  = np.random.default_rng(SEED)
n_te = len(y_te_bin)

boot_metrics = {'AUC': [], 'Recall': [], 'Precision': [], 'F1': []}

for _ in range(B):
    idx    = rng.integers(0, n_te, n_te)
    yb_bin = y_te_bin[idx]
    pb     = y_proba_dropout[idx]
    predb  = (pb >= opt_thresh).astype(int)

    if yb_bin.sum() < 2 or (1 - yb_bin).sum() < 2:
        continue
    try:
        boot_metrics['AUC'].append(roc_auc_score(yb_bin, pb))
        boot_metrics['Recall'].append(recall_score(yb_bin, predb, zero_division=0))
        boot_metrics['Precision'].append(precision_score(yb_bin, predb, zero_division=0))
        boot_metrics['F1'].append(f1_score(yb_bin, predb, zero_division=0))
    except Exception:
        pass

print('=== Bootstrap CI (B=1,000) — 95% percentil ===\n')
print(f'{"Métrica":<12} {"Media":>8} {"IC 95% inferior":>16} {"IC 95% superior":>16}')
print('-'*55)
ci_results = {}
for metric, vals in boot_metrics.items():
    vals = np.array(vals)
    lo, hi = np.percentile(vals, [2.5, 97.5])
    ci_results[metric] = (vals.mean(), lo, hi)
    print(f'{metric:<12} {vals.mean():>8.4f} {lo:>16.4f} {hi:>16.4f}')

print(f'\nIteraciones completadas: {len(boot_metrics["AUC"])}/{B}')

### 4.4 Matriz de Confusión y Gráfico de Métricas

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# ── Matriz de confusión ──
ax = axes[0]
cm = confusion_matrix(y_te_bin, y_pred_bin, normalize='true')
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Retenido','Desertor'])
disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='.2f')
ax.set_title(f'Matriz de Confusión (normalizada)\nAUC={auc_val:.3f}')

# ── Métricas con CI ──
ax = axes[1]
metric_names = list(ci_results.keys())
means  = [ci_results[m][0] for m in metric_names]
lowers = [ci_results[m][0] - ci_results[m][1] for m in metric_names]
uppers = [ci_results[m][2] - ci_results[m][0] for m in metric_names]

colors = ['#2563eb','#16a34a','#d97706','#dc2626']
bars = ax.bar(metric_names, means, yerr=[lowers, uppers],
              capsize=5, color=colors, alpha=0.8, ecolor='black')
for bar, val in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
ax.set_ylim(0, 1.15)
ax.set_ylabel('Valor')
ax.set_title('Métricas del Modelo con IC 95% (Bootstrap, B=1,000)')

plt.suptitle('Validación del Random Forest (PreTec21 → Tec21)', fontsize=12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_model_validation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✓ fig_model_validation.png guardado')

### 4.5 Análisis de Falsos Negativos

In [ ]:
# Falsos negativos: desertores (y_te_bin=1) no detectados (y_pred_bin=0)
fn_mask = (y_te_bin == 1) & (y_pred_bin == 0)
tp_mask = (y_te_bin == 1) & (y_pred_bin == 1)

print(f'Total desertores en test : {y_te_bin.sum():,}')
print(f'Verdaderos positivos (TP): {tp_mask.sum():,}')
print(f'Falsos negativos (FN)    : {fn_mask.sum():,}  ({fn_mask.sum()/y_te_bin.sum():.1%} de desertores)')

# Comparar features FN vs TP (usando matriz X_te ya escalada)
X_fn = X_te[fn_mask]
X_tp = X_te[tp_mask]

print('\n=== Diferencias de Features: FN vs TP ===\n')
print(f'{"Feature":<28} {"FN media":>10} {"TP media":>10} {"Δ":>10} {"p":>10}')
print('-'*65)

diffs = []
for i, feat in enumerate(feat_names):
    fn_v = X_fn[:, i]
    tp_v = X_tp[:, i]
    try:
        _, p = mannwhitneyu(fn_v, tp_v, alternative='two-sided')
    except Exception:
        p = 1.0
    delta = fn_v.mean() - tp_v.mean()
    diffs.append({'feat': feat, 'fn_mean': fn_v.mean(), 'tp_mean': tp_v.mean(),
                  'delta': delta, 'p': p})

df_diff = pd.DataFrame(diffs).sort_values('delta', key=abs, ascending=False)
for _, row in df_diff.head(8).iterrows():
    sig = '***' if row['p'] < 0.001 else ('**' if row['p'] < 0.01 else ('*' if row['p'] < 0.05 else 'ns'))
    print(f'{row["feat"]:<28} {row["fn_mean"]:>10.3f} {row["tp_mean"]:>10.3f} {row["delta"]:>10.3f} {row["p"]:>10.2e} {sig}')

---
## 5. Resumen de Resultados

In [ ]:
print('=' * 70)
print('  RESUMEN DE HALLAZGOS ESTADÍSTICOS')
print('=' * 70)

print('\n[I] NORMALIDAD — Ninguna variable sigue distribución normal (AD>crit. al 5%)')
print('    → Justifica uso de estadísticos no-paramétricos')

print('\n[II] DISTRIBUCIONES ÓPTIMAS (por AIC):')
for var, res in fit_results.items():
    best = res.iloc[0]
    print(f'    {var:<28} → {best["Distribución"]}  (AIC={best["AIC"]:.1f})')

print('\n[III] MANN-WHITNEY U (PreTec21 vs Tec21): diferencias significativas en todas las variables')
for var, r in mw_results.items():
    eff = 'grande' if abs(r['r_rb']) >= 0.5 else ('mediano' if abs(r['r_rb']) >= 0.3 else 'pequeño')
    print(f'    {var:<28} r_rb={r["r_rb"]:+.4f} ({eff})')

print('\n[IV] MODELO RF (PreTec21 → Tec21):')
for metric, (mean, lo, hi) in ci_results.items():
    print(f'    {metric:<12} {mean:.4f}  [IC 95%: {lo:.4f}–{hi:.4f}]')

print('\n' + '=' * 70)
print('Figuras guardadas en:', FIGURES_DIR.resolve())
print('=' * 70)